#### LLaVA 1.5 model on Transformers API with MH-Post_Hoc Framework
1. Loading LLaVA 1.5 model on GPU0
2. Loading MH-Detector on GPU1
3. Building the refinement loop and Ranking mechanism 
4. Sorting intermediate results, including generated content and similarity score, and final result
5. Converting the final result into Amber Json file format

In [1]:
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration, CLIPModel, CLIPProcessor

# Load LLaVA model on GPU0
llava_model = LlavaForConditionalGeneration.from_pretrained(
    "llava-hf/llava-1.5-7b-hf",
    torch_dtype=torch.float16,
    device_map={"": "cuda:0"}
)
llava_processor = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")

# Load CLIP model on GPU1
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to("cuda:1")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print("LLaVA Model loaded on GPU0")
print("CLIP Model loaded on GPU1")


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some kwargs in processor config are unused and will not have any effect: num_additional_image_tokens. 


LLaVA Model loaded on GPU0
CLIP Model loaded on GPU1


In [2]:
import os
import json
from PIL import Image
import requests
import random

# Path to the dataset and JSON file
dataset_dir = "/home/ubuntu/large_disk/project/MH_Post_Hoc/data/AMBER/image"
query_generative_json_path = "/home/ubuntu/large_disk/project/MH_Post_Hoc/AMBER/data/query/query_generative.json"

# Load the generative queries JSON file
with open(query_generative_json_path, "r") as f:
    query_generative_json = json.load(f)

# Check the number of queries in the query_generative.json
print("Number of queries in the query_generative.json: ", len(query_generative_json))

# Randomly select a few queries for testing purposes (e.g., 10)
random.seed(0)
selected_queries = random.sample(query_generative_json, 10)
print("Selected queries: ", selected_queries)

# Load images and prompts
images = []
prompts = []

for query in selected_queries:
    # Get the image path
    image_filename = query['image']
    image_path = os.path.join(dataset_dir, image_filename)

    # Load the image
    if os.path.exists(image_path):
        image = Image.open(image_path)
        images.append(image)
    else:
        print(f"Image not found: {image_path}")
        continue

    # Prepare the prompt
    prompt = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": query['query']},
            ],
        },
    ]
    prompts.append(prompt)

print(f"Loaded {len(images)} images and {len(prompts)} prompts.")

# Ensure the number of loaded images matches the number of prompts
assert len(images) == len(prompts), "Mismatch between number of images and prompts"

# Prepare prompts using LLaVA processor
processed_prompts = [llava_processor.apply_chat_template(prompt, add_generation_prompt=True) for prompt in prompts]

print("Prompts are processed and ready for generation.")


Number of queries in the query_generative.json:  1004
Selected queries:  [{'id': 865, 'image': 'AMBER_865.jpg', 'query': 'Describe this image.'}, {'id': 395, 'image': 'AMBER_395.jpg', 'query': 'Describe this image.'}, {'id': 777, 'image': 'AMBER_777.jpg', 'query': 'Describe this image.'}, {'id': 912, 'image': 'AMBER_912.jpg', 'query': 'Describe this image.'}, {'id': 431, 'image': 'AMBER_431.jpg', 'query': 'Describe this image.'}, {'id': 42, 'image': 'AMBER_42.jpg', 'query': 'Describe this image.'}, {'id': 266, 'image': 'AMBER_266.jpg', 'query': 'Describe this image.'}, {'id': 989, 'image': 'AMBER_989.jpg', 'query': 'Describe this image.'}, {'id': 524, 'image': 'AMBER_524.jpg', 'query': 'Describe this image.'}, {'id': 498, 'image': 'AMBER_498.jpg', 'query': 'Describe this image.'}]
Loaded 10 images and 10 prompts.
Prompts are processed and ready for generation.


In [14]:
import re

# Prepare inputs for generation using LLaVA processor
inputs = llava_processor(
    images=images,
    text=processed_prompts,
    padding=True,
    return_tensors="pt"
).to("cuda:0", torch.float16)  # Send inputs to GPU0

# Generate descriptions with the LLaVA model
with torch.no_grad():
    generated_ids = llava_model.generate(**inputs, max_new_tokens=50)

# Decode the generated descriptions
generated_descriptions = llava_processor.batch_decode(generated_ids, skip_special_tokens=True)

# Store the results (image and generated description pairs)
generated_results = []

# Regular expression to remove the templates like "USER: ..." and "ASSISTANT: ..."
template_pattern = re.compile(r"(USER:.*?ASSISTANT:)", re.DOTALL)

for query, description in zip(selected_queries, generated_descriptions):
    # Remove the prompt templates from the generated output
    cleaned_description = re.sub(template_pattern, "", description).strip()

    # Store only the cleaned response
    generated_results.append({
        "id": query["id"],
        "image": query["image"],
        "original_prompt": query["query"],
        "generated_description": cleaned_description
    })

# Print generated descriptions for verification
for result in generated_results:
    print(f"Image ID: {result['id']}, Description: {result['generated_description']}")

# Save the generated results into a JSON file for further processing
output_path = "/home/ubuntu/large_disk/project/MH_Post_Hoc/data/generated_results.json"
with open(output_path, "w") as f:
    json.dump(generated_results, f, indent=4)

print(f"Generated descriptions saved to {output_path}")


Image ID: 865, Description: The image features a large, fluffy cloud floating in a clear blue sky. The cloud is positioned towards the center of the sky, and its shape is reminiscent of a boat. The sky is filled with a few other clouds
Image ID: 395, Description: The image features a group of sheep grazing in a lush green field. There are at least nine sheep visible in the scene, with some standing closer to the foreground and others further in the background. The sheep are scattered throughout the field,
Image ID: 777, Description: The image features a black cat standing on a concrete floor, looking at a bowl of water. The cat appears to be curious about the water, possibly considering whether to drink from it. The bowl is placed on the floor, and the cat
Image ID: 912, Description: The image features a large boat sailing on a body of water, with a beautiful blue sky above. The boat is positioned in the middle of the scene, and it appears to be the main focus of the image.

In the for

In [30]:
from transformers import CLIPTextModel, CLIPTokenizer
import torch.nn.functional as F

# Prepare inputs for the CLIP model
clip_text_tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
clip_text_model = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32").to("cuda:1")

hallucination_results = []

for result in generated_results:
    # Load the image
    image_path = os.path.join(dataset_dir, result["image"])
    image = Image.open(image_path)

    # Process the image for CLIP model
    processed_image = clip_processor(images=image, return_tensors="pt").to("cuda:1")

    # Process the generated description
    text_inputs = clip_text_tokenizer(result["generated_description"], return_tensors="pt", padding=True).to("cuda:1")

    # Get the CLIP embeddings
    with torch.no_grad():
        image_features = clip_model.get_image_features(**processed_image)
        text_features = clip_text_model(**text_inputs).pooler_output

    # Normalize the features
    image_features = F.normalize(image_features, p=2, dim=-1)
    text_features = F.normalize(text_features, p=2, dim=-1)

    # Calculate similarity score
    similarity_score = torch.matmul(image_features, text_features.T).item()

    # Determine if there is hallucination
    threshold = 0.25  # This threshold can be adjusted based on your requirements
    hallucinated = similarity_score*100 < threshold

    # Store the results
    hallucination_results.append({
        "id": result["id"],
        "image": result["image"],
        "original_prompt": result["original_prompt"],
        "generated_description": result["generated_description"],
        "similarity_score": similarity_score,
        "hallucinated": hallucinated
    })

    # Print results for verification
    print(f"Image ID: {result['id']}, Similarity Score: {similarity_score}, Hallucinated: {hallucinated}")

# Save the hallucination check results to a JSON file
hallucination_output_path = "/home/ubuntu/large_disk/project/MH_Post_Hoc/data/hallucination_results.json"
with open(hallucination_output_path, "w") as f:
    json.dump(hallucination_results, f, indent=4)

print(f"Hallucination results saved to {hallucination_output_path}")


Image ID: 865, Similarity Score: 0.009466941468417645, Hallucinated: False
Image ID: 395, Similarity Score: 0.01687867008149624, Hallucinated: False
Image ID: 777, Similarity Score: -0.07651039212942123, Hallucinated: True
Image ID: 912, Similarity Score: -0.03400788828730583, Hallucinated: True
Image ID: 431, Similarity Score: -0.021236488595604897, Hallucinated: True
Image ID: 42, Similarity Score: 0.04351385310292244, Hallucinated: False
Image ID: 266, Similarity Score: 0.0008271578699350357, Hallucinated: True
Image ID: 989, Similarity Score: -0.024338021874427795, Hallucinated: True
Image ID: 524, Similarity Score: 0.03916780278086662, Hallucinated: False
Image ID: 498, Similarity Score: 0.06968849152326584, Hallucinated: False
Hallucination results saved to /home/ubuntu/large_disk/project/MH_Post_Hoc/data/hallucination_results.json


In [36]:
import re

# Set the maximum number of regeneration attempts
max_rounds = 3

# Choose the refinement strategy manually: "strategy_1" or "strategy_2"
refinement_strategy = "strategy_2"  # Change this to "strategy_1" for the alternative strategy

final_results = []

# Iterate over the hallucination results and regenerate if hallucinated
for result in hallucination_results:
    current_image_path = os.path.join(dataset_dir, result["image"])
    image = Image.open(current_image_path)
    original_prompt = result["original_prompt"]  # Original prompt to be appended
    current_prompt = result["generated_description"]  # This is the initial generated description
    similarity_scores = [result["similarity_score"]]
    descriptions = [current_prompt]
    hallucination_flags = [result["hallucinated"]]
    rounds_info = []

    hallucinated = result["hallucinated"]

    # Loop for regeneration up to max_rounds if hallucinated
    for round_num in range(max_rounds):
        # Define the refined prompt based on the selected strategy
        if refinement_strategy == "strategy_1":
            # Strategy 1: Prefix warning about object, attribute, and relationship hallucinations
            refined_prompt = (
                f"Please be careful of object, attribute, and relationship hallucinations. Now, "
                f"{original_prompt}"
            )
        elif refinement_strategy == "strategy_2":
            # Strategy 2: Feedback on the last response and prefix warning
            refined_prompt = (
                f"Your last response was: '{current_prompt}' and it may be inaccurate. "
                f"Please be careful of object, attribute, and relationship hallucinations in your response. "
                f"{original_prompt}"

            )
        elif refinement_strategy == "strategy_3":
            # Strategy 3: Prefix warning about object, attribute, and relationship hallucinations, with feedback on the generation process
            refined_prompt = (
                f"Please be careful of object, attribute, and relationship hallucinations. Now, "
                f"{original_prompt}"
            )
        else:
            raise ValueError("Invalid refinement strategy. Please choose 'strategy_1' or 'strategy_2'.")

        # Append the original prompt content to the refined prompt
        # refined_prompt += original_prompt
        
        # Construct prompt data for LLaVA model
        prompt_data = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": refined_prompt},
                ],
            },
        ]

        # Apply the chat template to create the input that will be sent to the model
        processed_prompt = llava_processor.apply_chat_template(prompt_data, add_generation_prompt=True)
        print(f"Refined prompt for round {round_num + 1}: {processed_prompt}")
        if refinement_strategy == "strategy_3":
            processed_prompt +=  f"The last response was: '{current_prompt}' and it is inaccurate. The new response is: "


        # Prepare inputs for LLaVA generation
        inputs = llava_processor(
            images=image,
            text=processed_prompt,
            padding=True,
            return_tensors="pt"
        ).to("cuda:0", torch.float16)

        # Generate new description
        with torch.no_grad():
            generated_ids = llava_model.generate(**inputs, max_new_tokens=50)
        regenerated_description = llava_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

        # Remove prompt template content from the regenerated description
        cleaned_description = re.sub(template_pattern, "", regenerated_description).strip()
        if refinement_strategy == "strategy_3":
            cleaned_description = cleaned_description.split("The new response is: ")[1]
        current_prompt = cleaned_description
        # Store the regenerated description and the current round information
        descriptions.append(cleaned_description)
        hallucination_flags.append(hallucinated)

        # Pass the image and regenerated description to CLIP for verification
        text_inputs = clip_text_tokenizer(
            current_prompt,
            return_tensors="pt",
            padding="max_length",
            max_length=77,  # Limit to the maximum length supported by CLIP
            truncation=True
        ).to("cuda:1")
        
        processed_image = clip_processor(images=image, return_tensors="pt").to("cuda:1")

        # Get similarity score from CLIP model
        with torch.no_grad():
            image_features = clip_model.get_image_features(**processed_image)
            text_features = clip_text_model(**text_inputs).pooler_output

        # Normalize the features
        image_features = F.normalize(image_features, p=2, dim=-1)
        text_features = F.normalize(text_features, p=2, dim=-1)

        # Calculate similarity score
        similarity_score = torch.matmul(image_features, text_features.T).item()
        similarity_scores.append(similarity_score)

        # Determine if hallucinated
        hallucinated = similarity_score < 0.25

        # Store current round info, including the input prompt and unprocessed output
        rounds_info.append({
            "round": round_num + 1,
            "input_prompt": processed_prompt,  # Store the final version of the input prompt after applying the template
            "original_output": regenerated_description,  # Store the original generated description without cleaning
            "description": cleaned_description,  # Store the cleaned description
            "similarity_score": similarity_score,
            "hallucinated": hallucinated
        })

        # Log the current round's result
        print(f"Round {round_num + 1}: Image ID: {result['id']}, Similarity Score: {similarity_score}, Hallucinated: {hallucinated}")

        # If no hallucination, stop further regeneration
        if not hallucinated:
            break

    # After max_rounds or successful verification, determine the final response
    if hallucinated:
        # Select the description with the highest similarity score if still hallucinated
        best_index = similarity_scores.index(max(similarity_scores))
        selection_reason = "highest_similarity_score"
    else:
        # If verified successfully, use the latest description
        best_index = len(descriptions) - 1
        selection_reason = "hallucination_detection_false"

    # Store the final result along with all the intermediate rounds information
    final_results.append({
        "id": result["id"],
        "image": result["image"],
        "initial_generated_prompt": result["generated_description"],
        "final_description": descriptions[best_index],
        "similarity_scores": similarity_scores,
        "hallucination_flags": hallucination_flags,
        "rounds_info": rounds_info,
        "selection_reason": selection_reason,
        "final_round_index": best_index  # Store the index of the round chosen as the final result
    })

# Save the final results into a JSON file
final_output_path = "/home/ubuntu/large_disk/project/MH_Post_Hoc/data/final_results_detailed.json"
with open(final_output_path, "w") as f:
    json.dump(final_results, f, indent=4)

print(f"Final results saved to {final_output_path}")


Refined prompt for round 1: USER: <image>
Your last response was: 'The image features a large, fluffy cloud floating in a clear blue sky. The cloud is positioned towards the center of the sky, and its shape is reminiscent of a boat. The sky is filled with a few other clouds' and it may be inaccurate. Please be careful of object, attribute, and relationship hallucinations in your response. Describe this image. ASSISTANT:
Round 1: Image ID: 865, Similarity Score: 0.003402112051844597, Hallucinated: True
Refined prompt for round 2: USER: <image>
Your last response was: 'The image showcases a beautiful blue sky with a large, fluffy cloud floating in the middle of it. The cloud is shaped like a boat and appears to be the main focus of the scene. The sky is clear and unobstruct' and it may be inaccurate. Please be careful of object, attribute, and relationship hallucinations in your response. Describe this image. ASSISTANT:
Round 2: Image ID: 865, Similarity Score: 0.003402112051844597, Hall

In [32]:
import json

# Load the hallucination results to inspect keys
hallucination_results_path = "/home/ubuntu/large_disk/project/MH_Post_Hoc/data/hallucination_results.json"
with open(hallucination_results_path, "r") as f:
    hallucination_results = json.load(f)

# Print the first result to inspect the keys
print(hallucination_results[0].keys())

dict_keys(['id', 'image', 'original_prompt', 'generated_description', 'similarity_score', 'hallucinated'])
